# Head Output Analysis

Per-head output decomposition: what each head writes to the residual stream, value-weighted source analysis, output direction characterization, head cooperation/competition, and output norm patterns.

## Why This Matters

Attention head output reveals how heads route information between positions. Understanding attention mechanics is central to reverse-engineering transformer circuits, as attention patterns determine what information each component can access and transform.

**Key references:**
- [Elhage et al. (2021) "A Mathematical Framework for Transformer Circuits"](https://transformer-circuits.pub/2021/framework/index.html) — Foundational framework for attention head and residual stream analysis
- [Olsson et al. (2022) "In-context Learning and Induction Heads"](https://arxiv.org/abs/2209.11895) — Discovers induction heads and training phase transitions

In [ ]:
import jax
import jax.numpy as jnp
import numpy as np
from irtk import HookedTransformer, HookedTransformerConfig
from irtk.head_output_analysis import (
    head_output_decomposition,
    value_weighted_analysis,
    output_direction_characterization,
    head_cooperation_competition,
    output_norm_analysis,
)

cfg = HookedTransformerConfig(
    n_layers=4, d_model=32, n_ctx=64, d_head=8, n_heads=4, d_vocab=100,
)
model = HookedTransformer(cfg)
key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = []
for leaf in leaves:
    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:
        key, subkey = jax.random.split(key)
        new_leaves.append(jax.random.normal(subkey, leaf.shape) * 0.1)
    else:
        new_leaves.append(leaf)
model = jax.tree.unflatten(treedef, new_leaves)
tokens = jnp.array([1, 15, 30, 45, 60, 75, 90])
print('Model ready')

In [ ]:
# Decompose each head's output contribution
result = head_output_decomposition(model, tokens, layer=0)
print(f'Number of heads: {result["n_heads"]}')
for h in result['per_head']:
    print(f'  Head {h["head"]}: norm={h["output_norm"]:.4f}, '
          f'promoted={h["promoted"][:2]}, demoted={h["demoted"][:2]}')

In [ ]:
# Value-weighted analysis: which source positions contribute most
result = value_weighted_analysis(model, tokens, layer=0, head=0)
print(f'Attention weights: {np.array(result["attention_weights"]).round(3)}')
print(f'Dominant source position: {result["dominant_source"]}')
for src in result['per_source_contribution'][:3]:
    print(f'  Pos {src["position"]}: weight={src["weight"]:.4f}, norm={src["contribution_norm"]:.4f}')

In [ ]:
# Characterize diversity of head output directions
result = output_direction_characterization(model, tokens, layer=0)
print(f'Diversity score: {result["diversity_score"]:.4f}')
print(f'Pairwise cosines shape: {result["pairwise_cosines"].shape}')
print('Pairwise cosine matrix:')
print(np.array(result['pairwise_cosines']).round(3))

In [ ]:
# Cooperation/competition between heads
result = head_cooperation_competition(model, tokens, layer=0)
print(f'Cooperation matrix shape: {result["cooperation_matrix"].shape}')
print(f'Cooperating pairs: {result["cooperating_pairs"][:3]}')
print(f'Competing pairs: {result["competing_pairs"][:3]}')

In [ ]:
# Output norm analysis across heads and positions
result = output_norm_analysis(model, tokens, layer=0)
print(f'Norm matrix shape: {result["norm_matrix"].shape}')
print(f'Head mean norms: {np.array(result["head_mean_norms"]).round(4)}')
print(f'Max norm head: {result["max_norm_head"]}')